<a href="https://colab.research.google.com/github/AktanM11/AI-OI/blob/main/DAY5_WEEK2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install fastembed

In [ ]:
pip install qdrant-client

In [ ]:
pip install sentence-transformers

In [18]:
import json

In [4]:
import qdrant_client
from qdrant_client import models
from qdrant_client import QdrantClient

In [29]:
from qdrant_client.models import VectorParams, PointStruct, Distance, SparseVectorParams, Prefetch
import sentence_transformers
from sentence_transformers import SentenceTransformer

In [9]:
from fastembed.sparse import SparseTextEmbedding

In [ ]:
model = SentenceTransformer('intfloat/multilingual-e5-large')

In [16]:
dense_size = model.get_sentence_embedding_dimension()

/tmp/ipykernel_2588/1148047738.py:1: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dense_size = model.get_sentence_embedding_dimension()


In [7]:
from google.colab import userdata
client = QdrantClient(url=userdata.get('QDRANT_URL'), api_key=userdata.get('QDRANT_API_KEY'))

In [ ]:
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")

In [8]:
file_path = "inventory.jsonl"
collection_name = "inventory_collection"

In [17]:
client.create_collection(
    collection_name=collection_name,
    # Configure Dense Vectors (named "dense-text")
    vectors_config={
        "dense-text": VectorParams(size=dense_size, distance=Distance.COSINE)
    },
    # Configure Sparse Vectors (named "sparse-text")
    sparse_vectors_config={
        "sparse-text": SparseVectorParams()
    }
)

True

In [ ]:
BATCH_SIZE = 50
total_uploaded = 0
with open(file_path, "r", encoding="utf-8") as f:
    points = []

    for idx, line in enumerate(f):
        if not line.strip():
            continue

        data = json.loads(line)

        search_text = data.get("search_text", "")

        dense_vector = model.encode(f"passage: {search_text}").tolist()

        sparse_output = list(sparse_model.embed([search_text]))[0]

        sparse_vector = {
            "indices": sparse_output.indices.tolist(),
            "values": sparse_output.values.tolist()
        }

        payload = {
            "product_name": data.get("product_name"),
            "brand": data.get("brand"),
            "store_name": data.get("store_name"),
            "store_city": data.get("store_city"),
            "in_stock": data.get("in_stock")
        }


        point = PointStruct(
            id=idx,
            vector={
                "dense-text": dense_vector,
                "sparse-text": sparse_vector
            },
            payload=payload
        )
        points.append(point)

        if len(points) >= BATCH_SIZE:
            client.upsert(collection_name=collection_name, points=points)
            total_uploaded += len(points)
            print(f"Uploaded batch of {len(points)} items. Total so far: {total_uploaded}")

            points = []

In [30]:
def hybrid_search(user_query: str):
    dense_query_vector = model.encode(f"query: {user_query}").tolist()

    sparse_output = list(sparse_model.embed([user_query]))[0]
    sparse_query_vector = {
        "indices": sparse_output.indices.tolist(),
        "values": sparse_output.values.tolist()
    }

    response = client.query_points(
        collection_name=collection_name,

        prefetch=[
            models.Prefetch(
                query=dense_query_vector,
                using="dense-text",
                limit=10
            ),

            models.Prefetch(
                query=sparse_query_vector,
                using="sparse-text",
                limit=10
            )
        ],

        query=models.FusionQuery(
            fusion=models.Fusion.RRF
        ),
        limit=3
    )

    if not response.points:
        print("No items found.")
        return

    for idx, hit in enumerate(response.points):
        print(f"\n[{idx+1}] RRF Score: {hit.score:.4f}")
        print(f"Product: {hit.payload.get('product_name')}")
        print(f"Brand: {hit.payload.get('brand')}")
        print(f"Location: {hit.payload.get('store_city')} ({hit.payload.get('store_name')})")
        print(f"In Stock: {hit.payload.get('in_stock')}")


In [36]:
hybrid_search("зарядка на айфон")
hybrid_search("BA52A")
hybrid_search("16 Pro Max")


[1] RRF Score: 0.5000
Product: Power Bank Borofone BJ79 10000mAh
Brand: Unknown
Location: Тюп (O!Store Тюп)
In Stock: True

[2] RRF Score: 0.3333
Product: Power Bank Borofone BJ79 10000mAh
Brand: Unknown
Location: Айдаркен (O!Store Айдаркен)
In Stock: True

[3] RRF Score: 0.2500
Product: Power Bank Borofone BJ79 10000mAh
Brand: Unknown
Location: Ивановка (O!Store Ивановка)
In Stock: True

[1] RRF Score: 0.5000
Product: ЗУ Borofone BA52A Type-C
Brand: Unknown
Location: Баткен (O!Store Баткен)
In Stock: True

[2] RRF Score: 0.5000
Product: ЗУ Borofone BA52A Lightning
Brand: Unknown
Location: Токмок (O!Store Токмок)
In Stock: True

[3] RRF Score: 0.3333
Product: ЗУ Borofone BA52A Micro
Brand: Unknown
Location: Токмок (O!Store Токмок)
In Stock: True

[1] RRF Score: 0.8333
Product: Apple iPhone 16 Pro Max 256 GB White Titanium
Brand: Apple
Location: Талас (O!Store Талас)
In Stock: True

[2] RRF Score: 0.7000
Product: Apple iPhone 16 Pro Max 256 GB Natural Titanium
Brand: Apple
Location: Би